In [4]:
import json
from pathlib import Path

import numpy as np

from experiments._common import H4_FCIDUMP_PATH
from qcmps import (
    Ansatz,
    FCIDumpLoader,
    GlobalQCMPS,
    HFPreset,
    Logger,
    Observable,
    SO4Gate,
    StarLayerType,
    BrickWallLayerType,
    TwoSiteOptimizer,
)

In [2]:
output_dir = Path.cwd() / "output"

logger = Logger(output_dir / f"h4_star_opt.log", value_file=None, continue_logging=False)

In [3]:
loader = FCIDumpLoader(H4_FCIDUMP_PATH)
loader.interleave_spin_orbitals()

n_phys_qubits = loader.n_spin_orbitals
n_bond_qubits = 2
n_qubits = n_phys_qubits + n_bond_qubits
physical_qubits = list(range(n_phys_qubits))

In [5]:
ansatz = Ansatz(n_qubits)
hf_preset = HFPreset(n_phys_qubits, loader.hf_occupancy())

qcmps = GlobalQCMPS(n_phys_qubits, n_bond_qubits, [StarLayerType(SO4Gate)])

ansatz.add_component(hf_preset, list(range(n_bond_qubits, n_qubits)))
ansatz.add_component(qcmps, list(range(n_qubits)))

ansatz.load_checkpoint(output_dir / "h4_star_1_5_check.json")

In [6]:
hamiltonian = Observable(loader.hamiltonian, physical_qubits, constant=loader.nuclear_repulsion, label="Total energy")
particle_number = Observable.particle_number(physical_qubits)

In [7]:
logger.log(f"Total energy: {hamiltonian(ansatz):.12f}")
logger.log(f"Energy variance: {hamiltonian.var(ansatz):.12f}")
logger.log(f"Particle number: {particle_number(ansatz):.12f}")
logger.log(f"Particle number variance: {particle_number.var(ansatz):.12f}")

Total energy: -2.232399263315
Energy variance: 0.051536106589
Particle number: 3.999999408959
Particle number variance: 0.000007170114


In [8]:
min_value = -0.2
max_value = 0.2

qcmps.add_layers([BrickWallLayerType(SO4Gate)], initializer = lambda shape: np.random.uniform(min_value, max_value, size=shape))
ansatz.rebuild()

In [19]:
min_value = 0
max_value = 0

qcmps.add_layers([StarLayerType(SO4Gate)], initializer = lambda shape: np.random.uniform(min_value, max_value, size=shape))
ansatz.rebuild()

In [20]:
logger.log(f"Total energy: {hamiltonian(ansatz):.12f}")
logger.log(f"Energy variance: {hamiltonian.var(ansatz):.12f}")
logger.log(f"Particle number: {particle_number(ansatz):.12f}")
logger.log(f"Particle number variance: {particle_number.var(ansatz):.12f}")

Total energy: -2.232442931658
Energy variance: 0.051243460561
Particle number: 3.999997922608
Particle number variance: 0.000006524958


In [29]:
new_layer_groups = qcmps.layer_parameter_groups(-1)
old_layer_groups = qcmps.layer_parameter_groups(0)

In [22]:
new_layer_optimizer = TwoSiteOptimizer(
    logger,
    log_every_n_steps=None,
    max_sweeps=3,
    local_maxiter=10,
    tolerance=1e-8,
)

result = new_layer_optimizer.optimize(ansatz, hamiltonian, qcmps, active_parameters = new_layer_groups)

[Optimizer] Sweep 1/3 started.
[Optimizer] [Sweep 1 Site pair 1-2] Total energy: -2.232442938439
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.232442938625
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.232442941256
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.232442942575
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.232442942783
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.232442946200
[Optimizer] [Sweep 1 Site pair 7-8] Total energy: -2.232442951082
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.232442953529
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.232442953820
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.232443087929
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.232443102338
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.232443103199
[Optimizer] [Sweep 1] Total energy: -2.232443103199, Relative change: 7.684e-08
[Optimizer] Sweep 2/3 started.
[Optimizer] [Sweep 2 Site pair 1-2] Total energy: -2.232443105647


In [11]:
old_layer_optimizer = TwoSiteOptimizer(
    logger,
    log_every_n_steps=None,
    max_sweeps=1,
    local_maxiter=1,
    tolerance=1e-8,
)

result = old_layer_optimizer.optimize(ansatz, hamiltonian, qcmps, active_parameters = old_layer_groups)

[Optimizer] Sweep 1/1 started.
[Optimizer] [Sweep 1 Site pair 1-2] Total energy: -1.703831249180
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -1.739868553941
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -1.936616605212
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -1.964159874320
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -1.970712392058
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.017538818427
[Optimizer] [Sweep 1 Site pair 7-8] Total energy: -2.043431500001
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.046475056470
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.048669762719
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.057789619530
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.076041014876
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.082534611337
[Optimizer] [Sweep 1] Total energy: -2.082534611337, Relative change: 2.766e-01


In [23]:
full_optimizer = TwoSiteOptimizer(
    logger,
    log_every_n_steps=None,
    max_sweeps=10,
    local_maxiter=10,
    tolerance=1e-8,
)

result = full_optimizer.optimize(ansatz, hamiltonian, qcmps)

[Optimizer] Sweep 1/10 started.
[Optimizer] [Sweep 1 Site pair 1-2] Total energy: -2.232443681802
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.232443684949
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.232443687040
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.232443688534
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.232443691656
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.232443693351
[Optimizer] [Sweep 1 Site pair 7-8] Total energy: -2.232443697181
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.232443700012
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.232443702286
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.232443702855
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.232443703578
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.232443704983
[Optimizer] [Sweep 1] Total energy: -2.232443704983, Relative change: 1.211e-08
[Optimizer] Sweep 2/10 started.
[Optimizer] [Sweep 2 Site pair 1-2] Total energy: -2.23244370787

In [25]:
class EnergyVarianceObjective:
    label = "Energy + variance"

    def __init__(self, hamiltonian, strength):
        self.hamiltonian = hamiltonian
        self.strength = strength

    def __call__(self, ansatz):
        return (
            self.hamiltonian(ansatz)
            + self.strength * self.hamiltonian.var(ansatz)
        )

In [27]:
class FastEnergyVarianceObjective:
    label = "Energy + variance"

    def __init__(
        self,
        hamiltonian_matrix,
        nuclear_repulsion,
        n_phys_qubits,
        n_bond_qubits,
        strength,
    ):
        self.hamiltonian_matrix = hamiltonian_matrix
        self.nuclear_repulsion = nuclear_repulsion
        self.n_phys_qubits = n_phys_qubits
        self.n_bond_qubits = n_bond_qubits
        self.strength = strength

    def __call__(self, ansatz: Ansatz):
        state = ansatz.statevector().data

        # Jeden řádek pro každý stav bond registru.
        branches = state.reshape(
            2**self.n_bond_qubits,
            2**self.n_phys_qubits,
        )

        h_branches = branches @ self.hamiltonian_matrix.T

        electronic_energy = np.real(
            np.vdot(branches, h_branches)
        )

        energy_squared = np.real(
            np.vdot(h_branches, h_branches)
        )

        variance = (
            energy_squared
            - electronic_energy**2
        )

        total_energy = (
            electronic_energy
            + self.nuclear_repulsion
        )

        return (
            total_energy
            + self.strength * variance
        )

In [28]:
hamiltonian_matrix = np.asarray(
    loader.hamiltonian.to_matrix(),
    dtype=complex,
)

In [ ]:
for strength in [1, 0.5, 0.03, 0.01]:
    objective = FastEnergyVarianceObjective(
        hamiltonian_matrix=hamiltonian_matrix,
        nuclear_repulsion=loader.nuclear_repulsion,
        n_phys_qubits=n_phys_qubits,
        n_bond_qubits=n_bond_qubits,
        strength=strength,
    )

    optimizer = TwoSiteOptimizer(
        logger,
        max_sweeps=5,
        local_maxiter=10,
        tolerance=1e-8,
    )

    optimizer.optimize(
        ansatz,
        objective,
        qcmps,
        active_parameters = None
    )

    logger.log(
        f"lambda={strength}: "
        f"E={hamiltonian(ansatz):.12f}, "
        f"Var(H)={hamiltonian.var(ansatz):.12f}"
    )
    
optimizer = TwoSiteOptimizer(
    logger,
    max_sweeps=5,
    local_maxiter=10,
    tolerance=1e-8,
)

optimizer.optimize(
    ansatz,
    hamiltonian,
    qcmps,
    active_parameters = None
)

[Optimizer] Sweep 1/5 started.
[Optimizer] [Sweep 1 Site pair 1-2] Energy + variance: -2.182362904678
[Optimizer] [Sweep 1 Site pair 2-3] Energy + variance: -2.183517433494
[Optimizer] [Sweep 1 Site pair 3-4] Energy + variance: -2.183568457239
[Optimizer] [Sweep 1 Site pair 4-5] Energy + variance: -2.183576241735
[Optimizer] [Sweep 1 Site pair 5-6] Energy + variance: -2.183578246270
[Optimizer] [Sweep 1 Site pair 6-7] Energy + variance: -2.183578806144
[Optimizer] [Sweep 1 Site pair 7-8] Energy + variance: -2.183579080111
[Optimizer] [Sweep 1 Site pair 6-7] Energy + variance: -2.183579192817
[Optimizer] [Sweep 1 Site pair 5-6] Energy + variance: -2.183579373496
[Optimizer] [Sweep 1 Site pair 4-5] Energy + variance: -2.183583317627
[Optimizer] [Sweep 1 Site pair 3-4] Energy + variance: -2.183586614962
[Optimizer] [Sweep 1 Site pair 2-3] Energy + variance: -2.183587281783
[Optimizer] [Sweep 1] Energy + variance: -2.183587281783, Relative change: 1.073e-03
[Optimizer] Sweep 2/5 started.
[

In [39]:
ansatz.save_checkpoint(output_dir / f"h4_star_opt_check.json")